In [6]:
import pandas as pd
import numpy as np
from datetime import datetime
import os

os.chdir(r'D:\BI_Project')

print("="*70)
print("DATA CLEANING PIPELINE FOR POWER BI - WITH RETURNS")
print("="*70)

# ============================================
# 1. LOAD RAW DATA
# ============================================

print("\n📂 Step 1: Loading Raw Data...")
df_raw = pd.read_csv('data/raw/OnlineRetail.csv', encoding='ISO-8859-1')
print(f"✅ Loaded: {len(df_raw):,} rows")

df = df_raw.copy()

# ============================================
# 2. INITIAL DATA AUDIT
# ============================================

print("\n📊 Step 2: Initial Data Audit...")
print(f"  • Total records: {len(df):,}")
print(f"  • Duplicates: {df.duplicated().sum():,}")
print(f"  • Missing CustomerID: {df['CustomerID'].isnull().sum():,}")
print(f"  • 'C' prefix invoices: {len(df[df['InvoiceNo'].astype(str).str.startswith('C', na=False)]):,}")
print(f"  • Negative quantities: {len(df[df['Quantity'] < 0]):,}")

# ============================================
# 3. DATA CLEANING STEPS
# ============================================

print("\n🧹 Step 3: Cleaning Data...")

# 3.1 Remove exact duplicates
before_dup = len(df)
df = df.drop_duplicates()
after_dup = len(df)
print(f"  ✓ Removed {before_dup - after_dup:,} duplicate rows")

# 3.2 Convert date to datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')
df = df[df['InvoiceDate'].notna()]
print(f"  ✓ Converted InvoiceDate to datetime")

# 3.3 DON'T separate cancellations - keep everything together!
# Instead, just label the transaction types

# Label C-prefix invoices
df['HasCPrefix'] = df['InvoiceNo'].astype(str).str.startswith('C', na=False)

# Calculate revenue BEFORE filtering
df['Revenue'] = df['Quantity'] * df['UnitPrice']

# 3.4 Remove only rows with ZERO price (but keep negative prices for returns)
before_price = len(df)
df = df[df['UnitPrice'] != 0]  # Changed from > 0 to != 0
after_price = len(df)
print(f"  ✓ Removed {before_price - after_price:,} rows with zero price (kept negative prices)")

# 3.5 Remove records without description
before_desc = len(df)
df = df[df['Description'].notna()]
after_desc = len(df)
print(f"  ✓ Removed {before_desc - after_desc:,} rows with missing descriptions")

# 3.6 Clean text fields - UPPERCASE for consistency
df['Description'] = df['Description'].str.strip().str.upper()
df['Country'] = df['Country'].str.strip().str.title()
df['StockCode'] = df['StockCode'].astype(str).str.strip().str.upper()
print(f"  ✓ Standardized text fields")

# 3.7 Classify transaction types
df['TransactionType'] = 'Sale'  # Default

# Returns/Cancellations: Any transaction with negative quantity OR C prefix OR negative price
df.loc[(df['Quantity'] < 0) | (df['HasCPrefix'] == True) | (df['UnitPrice'] < 0), 'TransactionType'] = 'Return'

# Alternative: More granular classification
df['IsCancellation'] = df['HasCPrefix']
df['IsReturn'] = (df['Quantity'] < 0) | (df['UnitPrice'] < 0)

print(f"\n  📊 Transaction Classification:")
print(f"  • Sales (positive): {len(df[df['TransactionType'] == 'Sale']):,}")
print(f"  • Returns/Cancellations: {len(df[df['TransactionType'] == 'Return']):,}")
print(f"  • C-prefix invoices: {df['HasCPrefix'].sum():,}")
print(f"  • Negative quantities: {(df['Quantity'] < 0).sum():,}")
print(f"  • Negative prices: {(df['UnitPrice'] < 0).sum():,}")

# ============================================
# 4. CREATE CALCULATED COLUMNS
# ============================================

print("\n➕ Step 4: Creating Calculated Columns...")

# Revenue already calculated in step 3.3
df['AbsoluteRevenue'] = df['Revenue'].abs()

# Verify revenue
print(f"\n  📊 Revenue Verification:")
positive_revenue = df[df['Revenue'] > 0]['Revenue'].sum()
negative_revenue = df[df['Revenue'] < 0]['Revenue'].sum()
net_revenue = df['Revenue'].sum()
return_count = len(df[df['Revenue'] < 0])

print(f"  • Positive revenue (sales): £{positive_revenue:,.2f}")
print(f"  • Negative revenue (returns): £{negative_revenue:,.2f}")
print(f"  • Net revenue: £{net_revenue:,.2f}")
print(f"  • Return transactions: {return_count:,}")

# Date components
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['MonthName'] = df['InvoiceDate'].dt.strftime('%B')
df['Quarter'] = df['InvoiceDate'].dt.quarter
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()
df['Date'] = df['InvoiceDate'].dt.date
df['Hour'] = df['InvoiceDate'].dt.hour
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M').astype(str)

# Customer segment
df['CustomerSegment'] = df['CustomerID'].notna().map({
    True: 'Registered',
    False: 'Guest'
})

# Product category
df['ProductCategory'] = df['StockCode'].astype(str).str[:2]

print(f"  ✓ Created calculated columns")

# Final verification
final_returns = df[df['Revenue'] < 0]
print(f"\n  ✅ Final verification: Dataset contains {len(final_returns):,} return transactions")

if len(final_returns) == 0:
    print("  ⚠️  WARNING: No returns in final dataset!")
else:
    return_rate = (len(final_returns) / len(df)) * 100
    print(f"  ✅ Return rate: {return_rate:.2f}%")

# Rename df to df_sales for consistency with rest of pipeline
df_sales = df.copy()

# ============================================
# 5. CREATE DIMENSION TABLES
# ============================================

print("\n⭐ Step 5: Creating Dimension Tables...")

# 5.1 Customer Dimension
df_customers = df_sales[df_sales['CustomerID'].notna()][['CustomerID', 'Country']].drop_duplicates()
df_customers['CustomerID'] = df_customers['CustomerID'].astype(int)
df_customers = df_customers.groupby('CustomerID').first().reset_index()
df_customers = df_customers.sort_values('CustomerID').reset_index(drop=True)
print(f"  ✓ Customer Dimension: {len(df_customers):,} customers")

# 5.2 Product Dimension
df_products_raw = df_sales[['StockCode', 'Description']].copy()
df_products_raw = df_products_raw.dropna(subset=['StockCode', 'Description'])
df_products_raw['StockCode'] = df_products_raw['StockCode'].astype(str).str.strip().str.upper()
df_products_raw['Description'] = df_products_raw['Description'].astype(str).str.strip().str.upper()
df_products = df_products_raw.drop_duplicates(subset=['StockCode'], keep='first')
df_products = df_products.reset_index(drop=True)
df_products['ProductID'] = range(1, len(df_products) + 1)
df_products = df_products[['ProductID', 'StockCode', 'Description']]
df_products = df_products.sort_values('StockCode').reset_index(drop=True)
print(f"  ✓ Product Dimension: {len(df_products):,} unique products")

# 5.3 Date Dimension
date_range = pd.date_range(
    start=df_sales['InvoiceDate'].min(),
    end=df_sales['InvoiceDate'].max(),
    freq='D'
)
df_dates = pd.DataFrame({
    'Date': date_range.date,
    'FullDate': date_range,
    'Year': date_range.year,
    'Quarter': date_range.quarter,
    'Month': date_range.month,
    'MonthName': date_range.strftime('%B'),
    'Day': date_range.day,
    'DayOfWeek': date_range.day_name(),
    'WeekOfYear': date_range.isocalendar().week,
    'IsWeekend': date_range.dayofweek.isin([5, 6]),
    'YearMonth': date_range.to_period('M').astype(str)
})
print(f"  ✓ Date Dimension: {len(df_dates):,} dates")

# 5.4 Country Dimension
df_countries = df_sales[['Country']].drop_duplicates()
df_countries['CountryID'] = range(1, len(df_countries) + 1)
df_countries = df_countries.sort_values('Country').reset_index(drop=True)
print(f"  ✓ Country Dimension: {len(df_countries):,} countries")

# ============================================
# 6. CREATE FACT TABLE
# ============================================

print("\n📈 Step 6: Creating Fact Table...")

df_fact = df_sales[[
    'InvoiceNo', 'InvoiceDate', 'StockCode', 'CustomerID',
    'Quantity', 'UnitPrice', 'Revenue', 'AbsoluteRevenue',
    'Country', 'TransactionType', 'IsCancellation', 'IsReturn',
    'Year', 'Month', 'MonthName', 'Quarter', 'DayOfWeek',
    'Date', 'Hour', 'YearMonth', 'CustomerSegment'
]].copy()

df_fact['CustomerID'] = df_fact['CustomerID'].fillna(0).astype(int)

print(f"  ✓ Fact Table: {len(df_fact):,} transactions")
print(f"  ✓ Returns in Fact Table: {len(df_fact[df_fact['Revenue'] < 0]):,}")

# ============================================
# 7. CREATE SUMMARY TABLES
# ============================================

print("\n📊 Step 7: Creating Summary Tables...")

# Monthly Summary
df_monthly = df_fact.groupby(['Year', 'Month', 'MonthName', 'YearMonth']).agg({
    'Revenue': 'sum',
    'InvoiceNo': 'nunique',
    'Quantity': 'sum'
}).reset_index()
df_monthly.columns = ['Year', 'Month', 'MonthName', 'YearMonth', 'TotalRevenue', 'TotalInvoices', 'TotalQuantity']
df_monthly = df_monthly.sort_values(['Year', 'Month'])
print(f"  ✓ Monthly Summary: {len(df_monthly)} months")

# Country Summary
df_country_summary = df_fact.groupby('Country').agg({
    'Revenue': 'sum',
    'InvoiceNo': 'nunique',
    'CustomerID': lambda x: x[x > 0].nunique()
}).reset_index()
df_country_summary.columns = ['Country', 'TotalRevenue', 'TotalInvoices', 'UniqueCustomers']
df_country_summary = df_country_summary.sort_values('TotalRevenue', ascending=False)
print(f"  ✓ Country Summary: {len(df_country_summary)} countries")

# Product Performance
df_product_perf = df_fact.groupby('StockCode').agg({
    'Revenue': 'sum',
    'Quantity': 'sum',
    'InvoiceNo': 'nunique'
}).reset_index()
df_product_perf.columns = ['StockCode', 'TotalRevenue', 'TotalQuantity', 'TimesOrdered']
df_product_perf = df_product_perf.sort_values('TotalRevenue', ascending=False)
df_product_perf = df_product_perf.merge(df_products[['StockCode', 'Description']], on='StockCode', how='left')
print(f"  ✓ Product Performance: {len(df_product_perf)} products")

# Daily Sales
df_daily = df_fact.copy()
df_daily['Date'] = pd.to_datetime(df_daily['Date'])
df_daily = df_daily.groupby('Date').agg({
    'Revenue': 'sum',
    'InvoiceNo': 'nunique',
    'Quantity': 'sum'
}).reset_index()
df_daily.columns = ['Date', 'DailyRevenue', 'DailyInvoices', 'DailyQuantity']
print(f"  ✓ Daily Sales: {len(df_daily)} days")

# Hourly Pattern
df_hourly = df_fact.groupby('Hour').agg({
    'Revenue': 'sum',
    'InvoiceNo': 'nunique'
}).reset_index()
df_hourly.columns = ['Hour', 'HourlyRevenue', 'HourlyInvoices']
print(f"  ✓ Hourly Pattern: {len(df_hourly)} hours")

# ============================================
# 8. SAVE DATA
# ============================================

print("\n💾 Step 8: Saving Data...")

os.makedirs('data/powerbi', exist_ok=True)

tables_to_save = {
    'FactSales': df_fact,
    'DimCustomer': df_customers,
    'DimProduct': df_products,
    'DimDate': df_dates,
    'DimCountry': df_countries,
    'SummaryMonthly': df_monthly,
    'SummaryCountry': df_country_summary,
    'SummaryProduct': df_product_perf,
    'SummaryDaily': df_daily,
    'SummaryHourly': df_hourly
}

for table_name, table_df in tables_to_save.items():
    file_path = f'data/powerbi/{table_name}.csv'
    table_df.to_csv(file_path, index=False, encoding='utf-8-sig')
    print(f"  ✓ Saved: {table_name}.csv ({len(table_df):,} rows)")

# ============================================
# 9. GENERATE REPORT
# ============================================

print("\n📋 Step 9: Generating Report...")

report = f"""
{'='*80}
DATA CLEANING SUMMARY REPORT - WITH RETURNS INCLUDED
{'='*80}

Cleaning Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Project: Business Intelligence - E-Commerce Analysis
Students: Kartik Bhat (P23), Sanjay Yadav (P43)

{'='*80}
BEFORE CLEANING:
{'='*80}
Total Records:           {len(df_raw):,}
Duplicates:              {df_raw.duplicated().sum():,}
Missing CustomerID:      {df_raw['CustomerID'].isnull().sum():,}
Negative Quantities:     {len(df_raw[df_raw['Quantity'] < 0]):,}

{'='*80}
AFTER CLEANING:
{'='*80}
Total Records:           {len(df_fact):,}
Sales Transactions:      {len(df_fact[df_fact['TransactionType'] == 'Sale']):,}
Return Transactions:     {len(df_fact[df_fact['TransactionType'] == 'Return']):,}
Unique Customers:        {len(df_customers):,}
Unique Products:         {len(df_products):,}
Countries:               {len(df_countries):,}

{'='*80}
KEY METRICS:
{'='*80}
Gross Revenue (Sales):   £{positive_revenue:,.2f}
Returns (Negative):      £{negative_revenue:,.2f}
Net Revenue:             £{net_revenue:,.2f}
Return Rate:             {(abs(negative_revenue)/positive_revenue)*100:.2f}%

✅ RETURNS ARE INCLUDED IN THE DATASET
   Return transactions: {return_count:,}
   Return revenue: £{negative_revenue:,.2f}

{'='*80}
"""

print(report)

os.makedirs('reports', exist_ok=True)
with open('reports/DataCleaning_Report.txt', 'w', encoding='utf-8') as f:
    f.write(report)

print("✅ Report saved")

# ============================================
# 10. FINAL SUMMARY
# ============================================

print("\n" + "="*70)
print("✅ DATA CLEANING COMPLETE - RETURNS INCLUDED!")
print("="*70)
print(f"\n📊 Summary:")
print(f"  • Total transactions: {len(df_fact):,}")
print(f"  • Sales: {len(df_fact[df_fact['Revenue'] > 0]):,}")
print(f"  • Returns: {len(df_fact[df_fact['Revenue'] < 0]):,}")
print(f"\n💰 Revenue:")
print(f"  • Gross: £{positive_revenue:,.2f}")
print(f"  • Returns: £{negative_revenue:,.2f}")
print(f"  • Net: £{net_revenue:,.2f}")
print(f"  • Return Rate: {(abs(negative_revenue)/positive_revenue)*100:.2f}%")
print(f"\n🚀 Next: Refresh Power BI and check Return Rate %")
print("="*70)

if return_count > 0:
    print(f"\n✅ SUCCESS! Returns included: {return_count:,} transactions")
    print(f"   Expected Return Rate % in Power BI: ~{(abs(negative_revenue)/positive_revenue)*100:.1f}%")
else:
    print("\n⚠️  WARNING: No returns found!")

DATA CLEANING PIPELINE FOR POWER BI - WITH RETURNS

📂 Step 1: Loading Raw Data...
✅ Loaded: 541,909 rows

📊 Step 2: Initial Data Audit...
  • Total records: 541,909
  • Duplicates: 5,268
  • Missing CustomerID: 135,080
  • 'C' prefix invoices: 9,288
  • Negative quantities: 10,624

🧹 Step 3: Cleaning Data...
  ✓ Removed 5,268 duplicate rows
  ✓ Converted InvoiceDate to datetime
  ✓ Removed 2,510 rows with zero price (kept negative prices)
  ✓ Removed 0 rows with missing descriptions
  ✓ Standardized text fields

  📊 Transaction Classification:
  • Sales (positive): 524,878
  • Returns/Cancellations: 9,253
  • C-prefix invoices: 9,251
  • Negative quantities: 9,251
  • Negative prices: 2

➕ Step 4: Creating Calculated Columns...

  📊 Revenue Verification:
  • Positive revenue (sales): £10,642,110.80
  • Negative revenue (returns): £-916,103.85
  • Net revenue: £9,726,006.95
  • Return transactions: 9,253
  ✓ Created calculated columns

  ✅ Final verification: Dataset contains 9,253 retu